# Phase 0 + 1 + 2: Environment, Data Download & EEG Preprocessing

- **Phase 0**: GPU check, Drive mount, package install, directory tree
- **Phase 1**: Download THINGS-EEG2 EEG and image metadata from OSF
- **Phase 2**: Average repetitions, build filename manifest, sanity check

All artifacts write to Google Drive and are skipped if already present.

## Phase 0 — Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q torch torchvision transformers==4.44.0 mne==1.7.1 numpy scipy scikit-learn pandas matplotlib seaborn tqdm einops h5py pyyaml

In [ ]:
import os, json, torch

ROOT = "/content/drive/MyDrive/tribe-eeg"
WORK = "/content/work"

SUBDIRS = [
    "raw/thingseeg2_preproc", "raw/thingseeg2_metadata", "raw/alljoined",
    "processed", "embeddings",
    "checkpoints/linear_baseline", "checkpoints/per_subject_transformer",
    "checkpoints/multi_subject_transformer", "checkpoints/alljoined_transfer",
    "logs", "results", "figures",
]
for d in SUBDIRS:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
os.makedirs(WORK, exist_ok=True)

gpu_lines = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
env = {"gpu": gpu_lines[0] if gpu_lines else "none", "torch": torch.__version__}
with open(f"{ROOT}/logs/env.json", "w") as f:
    json.dump(env, f, indent=2)

print("Directories ready.")
print("GPU:", env["gpu"])
print("PyTorch:", env["torch"])

## Phase 1a — Download Preprocessed EEG from OSF

In [ ]:
import os, io, zipfile
from tqdm.notebook import tqdm

EEG_OSF_URL = "https://files.de-1.osf.io/v1/resources/anp5v/providers/osfstorage/?zip="
EEG_DEST = f"{ROOT}/raw/thingseeg2_preproc"
ZIP_PATH = "/content/thingseeg2_preproc.zip"
CHUNK = 64 * 1024 * 1024  # 64 MB — keeps per-file RAM writes bounded

os.makedirs(EEG_DEST, exist_ok=True)

# Clean up any partial unzip left on /content/ from a previous attempt
if os.path.isdir("/content/thingseeg2_preproc"):
    import shutil
    shutil.rmtree("/content/thingseeg2_preproc")
    print("Removed partial /content/thingseeg2_preproc.")

# Download outer zip to /content/ only if not already present
if not os.path.exists(ZIP_PATH):
    print("Downloading outer zip to /content (~51 GB, ~60 min)...")
    !wget -q --show-progress -O "{ZIP_PATH}" "{EEG_OSF_URL}"
else:
    print(f"Outer zip already present ({os.path.getsize(ZIP_PATH)/1e9:.1f} GB), skipping download.")

# Stream-extract: one inner subject zip at a time into RAM, write directly to Drive.
# Never writes inner zips to /content/ — peak /content usage stays at ~51 GB (the outer zip).
with zipfile.ZipFile(ZIP_PATH) as outer:
    all_names = outer.namelist()
    # Prefer 63-channel variants; fall back to all inner zips if not found
    inner_zips = sorted([n for n in all_names if n.endswith('.zip') and '63' in n])
    if not inner_zips:
        inner_zips = sorted([n for n in all_names if n.endswith('.zip')])
    print(f"Found {len(inner_zips)} inner zips to extract.")

    for inner_name in inner_zips:
        basename = os.path.basename(inner_name)
        sub_id = basename.split('_')[0]   # "sub-04__63_channels.zip" → "sub-04"
        sub_dest = f"{EEG_DEST}/{sub_id}"

        # Skip subject if .npy files already on Drive
        if os.path.exists(sub_dest):
            npy_files = [f for f in os.listdir(sub_dest) if f.endswith('.npy')]
            if npy_files:
                print(f"  {sub_id}: {len(npy_files)} .npy files already on Drive, skipping.")
                continue

        os.makedirs(sub_dest, exist_ok=True)
        print(f"  {sub_id}: reading inner zip into RAM (~4 GB)...", flush=True)
        inner_bytes = outer.open(inner_name).read()

        with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
            members = [m for m in inner.infolist() if not m.filename.endswith('/')]
            print(f"  {sub_id}: writing {len(members)} files to Drive...", flush=True)
            for member in tqdm(members, desc=sub_id, leave=False):
                out_path = os.path.join(sub_dest, os.path.basename(member.filename))
                with inner.open(member) as src, open(out_path, 'wb') as dst:
                    while True:
                        chunk = src.read(CHUNK)
                        if not chunk:
                            break
                        dst.write(chunk)

        del inner_bytes
        print(f"  {sub_id}: done.", flush=True)

# Free the 51 GB zip from /content/ now that everything is on Drive
os.remove(ZIP_PATH)
print("\nDone. Outer zip removed from /content/.")


## Phase 1b — Download Image Metadata from OSF

In [ ]:
IMG_OSF_URL = "https://files.de-1.osf.io/v1/resources/y63gw/providers/osfstorage/?zip="
IMG_DEST = f"{ROOT}/raw/thingseeg2_metadata"

if os.path.exists(f"{IMG_DEST}/training_images"):
    print("Image metadata already downloaded, skipping.")
else:
    print("Downloading image metadata. One-time operation.")
    !wget -q --show-progress -O /content/thingseeg2_metadata.zip "{IMG_OSF_URL}"
    !unzip -q /content/thingseeg2_metadata.zip -d "{IMG_DEST}"
    !cd "{IMG_DEST}" && unzip -q training_images.zip && unzip -q test_images.zip && rm training_images.zip test_images.zip
    !rm /content/thingseeg2_metadata.zip
    print("Done.")

## Phase 1c — Sanity Checks

In [ ]:
import os, json

inventory = {}
eeg_dir = f"{ROOT}/raw/thingseeg2_preproc"
subjects_found = sorted([d for d in os.listdir(eeg_dir) if d.startswith("sub-")])
inventory["n_subjects"] = len(subjects_found)
print(f"Subjects found: {len(subjects_found)}/10")

for sub in subjects_found:
    npy_files = [f for f in os.listdir(f"{eeg_dir}/{sub}") if f.endswith(".npy")]
    inventory[sub] = {"npy_files": npy_files}
    print(f"  {sub}: {npy_files}")

train_img_dir = f"{ROOT}/raw/thingseeg2_metadata/training_images"
test_img_dir  = f"{ROOT}/raw/thingseeg2_metadata/test_images"
n_train_concepts = len(os.listdir(train_img_dir))
n_test_concepts  = len(os.listdir(test_img_dir))
n_train_images   = sum(len(os.listdir(f"{train_img_dir}/{c}")) for c in os.listdir(train_img_dir))
n_test_images    = sum(len(os.listdir(f"{test_img_dir}/{c}")) for c in os.listdir(test_img_dir))

inventory.update({"train_concepts": n_train_concepts, "train_images": n_train_images,
                  "test_concepts": n_test_concepts, "test_images": n_test_images})
print(f"Training: {n_train_concepts} concepts, {n_train_images} images (expected 1654, 16540)")
print(f"Test:     {n_test_concepts} concepts, {n_test_images} images (expected 200, 200)")

with open(f"{ROOT}/logs/data_inventory.json", "w") as f:
    json.dump(inventory, f, indent=2)
print("Inventory saved.")

## Phase 2 — Inspect Raw EEG Format

Run this first to confirm filenames and shapes before the averaging loop.

In [ ]:
import numpy as np

sub01_dir = f"{ROOT}/raw/thingseeg2_preproc/sub-01"
channel_info = {}

for fname in sorted(os.listdir(sub01_dir)):
    if fname.endswith(".npy"):
        arr = np.load(f"{sub01_dir}/{fname}", mmap_mode="r")
        print(f"{fname}: shape={arr.shape}, dtype={arr.dtype}")
        if arr.ndim == 4:
            channel_info[fname] = {"n_channels": arr.shape[2], "shape": list(arr.shape)}

# Log which channel version we have — determines output head size for all downstream models
n_ch = list(channel_info.values())[0]["n_channels"] if channel_info else None
if n_ch == 63:
    print("\nChannel version: 63-channel BioSemi montage (full). Use this.")
elif n_ch == 17:
    print("\nChannel version: 17-channel occipital/parietal subset. Use this.")
else:
    print(f"\nUnexpected channel count: {n_ch}. Stop and investigate before continuing.")

channel_info["n_channels_used"] = n_ch
with open(f"{ROOT}/logs/channel_info.json", "w") as f:
    json.dump(channel_info, f, indent=2)
print("Channel info saved to logs/channel_info.json")


## Phase 2 — Average Repetitions

Update `TRAIN_FILE` / `TEST_FILE` if the filenames above differ from the defaults.

In [ ]:
import numpy as np
from tqdm import tqdm

PROC = f"{ROOT}/processed"
TRAIN_FILE = "preprocessed_eeg_training.npy"
TEST_FILE  = "preprocessed_eeg_test.npy"

def average_subject(sub_id, force=False):
    out_train = f"{PROC}/eeg_train_avg_{sub_id}.npz"
    out_test  = f"{PROC}/eeg_test_avg_{sub_id}.npz"
    if not force and os.path.exists(out_train) and os.path.exists(out_test):
        print(f"{sub_id}: already processed, skipping.")
        return
    sub_dir = f"{ROOT}/raw/thingseeg2_preproc/{sub_id}"
    train = np.load(f"{sub_dir}/{TRAIN_FILE}")  # (N_train, n_reps, C, T)
    test  = np.load(f"{sub_dir}/{TEST_FILE}")   # (N_test,  n_reps, C, T)
    print(f"{sub_id}: train {train.shape}, test {test.shape}")
    train_avg = train.mean(axis=1).astype(np.float32)
    test_avg  = test.mean(axis=1).astype(np.float32)
    assert not np.isnan(train_avg).any(), f"NaN in {sub_id} train"
    assert not np.isnan(test_avg).any(),  f"NaN in {sub_id} test"
    np.savez_compressed(out_train, eeg=train_avg)
    np.savez_compressed(out_test,  eeg=test_avg)
    print(f"  -> saved.")

for i in tqdm(range(1, 11)):
    average_subject(f"sub-{i:02d}")

## Phase 2 — Build Image Filename Manifest

In [ ]:
import os, json

def build_manifest(img_dir, expected_count, out_path):
    imgs = []
    for concept in sorted(os.listdir(img_dir)):
        concept_dir = f"{img_dir}/{concept}"
        if not os.path.isdir(concept_dir):
            continue
        for img in sorted(os.listdir(concept_dir)):
            imgs.append(f"{concept}/{img}")
    assert len(imgs) == expected_count, f"Expected {expected_count}, got {len(imgs)}"
    with open(out_path, "w") as f:
        json.dump(imgs, f)
    print(f"Saved {len(imgs)} filenames -> {out_path}")

build_manifest(
    f"{ROOT}/raw/thingseeg2_metadata/training_images",
    16540,
    f"{PROC}/image_filenames_train.json",
)
build_manifest(
    f"{ROOT}/raw/thingseeg2_metadata/test_images",
    200,
    f"{PROC}/image_filenames_test.json",
)

## Phase 2 — Evoked Response Sanity Check

Expect a clear peak at 100-200 ms post-stimulus. If flat, check EEG-image alignment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

test_eeg = np.load(f"{PROC}/eeg_test_avg_sub-01.npz")["eeg"]  # (200, C, T)
mean_response = test_eeg.mean(axis=(0, 1))  # (T,)
times_ms = np.linspace(-200, 800, mean_response.shape[0])

plt.figure(figsize=(8, 3))
plt.plot(times_ms, mean_response)
plt.axvline(0, color="k", linestyle="--", linewidth=0.8, label="stimulus onset")
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude (a.u.)")
plt.title("Mean evoked response - sub-01 test set")
plt.legend()
plt.tight_layout()
plt.savefig(f"{ROOT}/figures/sanity_evoked.png", dpi=150)
plt.show()
print("Success criterion: clear peak at 100-200 ms. If flat, check EEG-image alignment.")